# Data cleaning process

## ADNI MERGE

In [1]:
import pandas as pd
import numpy as np
from dl_client import DatalakeClient
import string

client = DatalakeClient()

### MONTHS

In [2]:
def convert_visitcode_to_int(value: string) -> int or string:
    """
    Convert the visitcode column to integer.
    """

    # If the value is 'sc' or 'f', return the value
    if value == 'sc' or value == 'f':
        return value
    
    # If the value is 'bl', return 0
    if value == 'bl':
        return 0

    # If the value is 'm', return the month
    if value[0] == 'm':
        return int(value.split('m')[1])
    
    raise ValueError('Ops, qualche caso non è stato considerato', value)

In [3]:
def handle_f_sc_values(df: pd.DataFrame, dataset: pd.DataFrame, column: str) -> pd.DataFrame:
    """
    Handle the 'f' and 'sc' values in the visitcode column.
    """
    # Verify column exists in both dataframes
    if column not in df.columns or column not in dataset.columns:
        raise ValueError(f"Column '{column}' not found in one or both dataframes")
    
    if 'PTID' not in df.columns or 'PTID' not in dataset.columns:
        raise ValueError("Column 'PTID' not found in one or both dataframes")
    
    # Create a copy of the dataframe to avoid modifying the original
    result_df = df.copy()
    
    # Track patients with errors for reporting
    patients_with_errors = []
    
    # Process each unique patient ID
    for ptid in result_df['PTID'].unique():
        # Get all visits for this patient
        patient_data = dataset[dataset['PTID'] == ptid]
        
        if len(patient_data) > 1:
            # Check if patient has both 'f' and 'sc'
            unique_values = patient_data[column].unique()
            has_f = 'f' in unique_values
            has_sc = 'sc' in unique_values
            
            if has_f and has_sc:
                patients_with_errors.append(ptid)
            
            # Update values for this patient in the result dataframe
            if has_f or has_sc:
                # Get indices for this patient in the result dataframe
                patient_indices = result_df.index[result_df['PTID'] == ptid]
                
                # Update those rows with value 0
                result_df.loc[patient_indices, column] = 0
    
    # Report errors if any
    if patients_with_errors:
        print(f"Errore: {len(patients_with_errors)} pazienti hanno sia 'f' che 'sc':")
        for ptid in patients_with_errors:
            print(f"  - Paziente: {ptid}")
        return None
    
    return result_df

In [4]:
def drop_if_all_none(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """
    Remove rows from dataframe where all values in the specified columns are None/NaN.
    """
    # Verify that all specified columns exist in the dataframe
    for col in columns:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found in dataframe")
    
    # Create a mask for rows where all specified columns are NaN
    mask = df[columns].isna().all(axis=1)
    
    # Return dataframe with those rows dropped
    return df[~mask].copy()

In [5]:
def handle_null_viscode2(df: pd.DataFrame, columns_to_check: list, date_column: str) -> pd.DataFrame:
    """
    Process rows where VISCODE2 is null:
    - If all values in specified columns are null, remove the row
    - Otherwise, calculate the correct month based on the previous visit date of the patient
    """
    # Create a copy of the dataframe to avoid modifying the original
    result_df = df.copy()
    
    # Verify that all required columns exist in the dataframe
    required_cols = columns_to_check + [date_column, 'VISCODE2', 'PTID']
    for col in required_cols:
        if col not in result_df.columns:
            raise ValueError(f"Column '{col}' not found in dataframe")
    
    # Convert date column to datetime if it isn't already
    result_df[date_column] = pd.to_datetime(result_df[date_column])
    
    # Get rows with null VISCODE2
    null_viscode_mask = result_df['VISCODE2'].isna()
    rows_to_process = result_df[null_viscode_mask].copy()
    
    # Create a mask for rows to drop (all values in columns_to_check are null)
    drop_mask = rows_to_process[columns_to_check].isna().all(axis=1)
    
    # Create a list to collect rows to keep with updated VISCODE2
    rows_to_update = []
    
    # Process rows where not all columns are null
    for idx, row in rows_to_process[~drop_mask].iterrows():
        patient_id = row['PTID']
        visit_date = row[date_column]
        
        # Get all visits for this patient, sorted by date
        patient_visits = result_df[result_df['PTID'] == patient_id].sort_values(by=date_column)
        
        # Find the previous visit (if any)
        prev_visits = patient_visits[patient_visits[date_column] < visit_date]
        
        if len(prev_visits) > 0:
            # Get the most recent previous visit
            prev_visit = prev_visits.iloc[-1]
            
            if pd.notna(prev_visit['VISCODE2']):
                # Calculate date difference in months
                date_diff = (visit_date - prev_visit[date_column]).days / 30.44  # Average days per month
                
                if prev_visit['VISCODE2'] == 'bl' or prev_visit['VISCODE2'] == 'sc' or prev_visit['VISCODE2'] == 'f': 
                    # If previous visit was baseline, calculate months since baseline
                    new_viscode = f"m{int(round(date_diff))}"
                else:
                    # If previous visit had mXX format, add the months
                    prev_month = int(prev_visit['VISCODE2'].replace('m', ''))
                    new_viscode = f"m{int(round(prev_month + date_diff))}"
                
                # Update the row's VISCODE2
                result_df.loc[idx, 'VISCODE2'] = new_viscode
            else:
                # If previous visit also had null VISCODE2, we can't reliably calculate
                # Keep the row but leave VISCODE2 as null
                pass
        else:
            # No previous visits, might be baseline
            result_df.loc[idx, 'VISCODE2'] = 'bl'
    
    # Filter out rows with all null values in specified columns and null VISCODE2
    rows_to_drop = rows_to_process[drop_mask].index
    result_df = result_df.drop(rows_to_drop)
    
    return result_df

In [6]:
def replace_unknown_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Replace 'Unknown', 'unknown', and '-4' values with NaN across all columns in a dataframe.
    """
    # Create a copy of the dataframe to avoid modifying the original
    result_df = df.copy()
    
    # Dictionary of values to replace with NaN
    replace_dict = {
        'Unknown': np.nan,
        'unknown': np.nan,
        '-4': np.nan
    }
    
    # Replace values across the entire dataframe
    result_df = result_df.replace(replace_dict)
    
    # Handle numeric columns where -4 might be stored as an integer
    for col in result_df.select_dtypes(include=['number']).columns:
        result_df[col] = result_df[col].replace(-4, np.nan)
    
    return result_df

### Apply to dataset

In [7]:
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'PTDEMOG'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

dataset = zip_files[list(zip_files.keys())[0]]

In [8]:
dataset

,PHASE,PTID,RID,VISCODE,VISCODE2,VISDATE,PTSOURCE,PTGENDER,PTDOB,PTDOBYY,...,PTBIRPR,PTBIRGR,ID,SITEID,USERDATE,USERDATE2,DD_CRF_VERSION_LABEL,LANGUAGE_CODE,HAS_QC_ERROR,update_stamp
0,ADNI1,011_S_0002,2,sc,sc,2005-08-17,1.0,1.0,04/1931,1931.0,...,NaN,NaN,18,107,2005-08-17,NaN,NaN,NaN,NaN,2005-08-17 00:00:00.0
1,ADNI1,022_S_0001,1,f,f,2005-08-18,1.0,2.0,12/1944,1944.0,...,NaN,NaN,20,10,2005-08-18,NaN,NaN,NaN,NaN,2005-08-18 00:00:00.0
2,ADNI1,011_S_0003,3,sc,sc,2005-08-18,1.0,1.0,05/1924,1924.0,...,NaN,NaN,22,107,2005-08-18,NaN,NaN,NaN,NaN,2005-08-18 00:00:00.0
3,ADNI1,022_S_0004,4,sc,sc,2005-08-18,1.0,1.0,01/1938,1938.0,...,NaN,NaN,24,10,2005-08-18,NaN,NaN,NaN,NaN,2005-08-18 00:00:00.0
4,ADNI1,011_S_0005,5,sc,sc,2005-08-23,1.0,1.0,12/1931,1931.0,...,NaN,NaN,26,107,2005-08-23,NaN,NaN,NaN,NaN,2005-08-23 00:00:00.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6042,ADNI4,082_S_10809,10809,4_sc,sc,2025-06-03,1.0,2.0,08/1945,1945.0,...,1.0,0.0,96846,82,2025-06-04,2025-06-04,v1,e,0.0,2025-06-05 01:44:21.0
6043,ADNI4,002_S_10814,10814,4_sc,sc,2025-06-02,1.0,1.0,03/1948,1948.0,...,1.0,0.0,96992,2,2025-06-04,2025-06-04,v1,e,1.0,2025-06-05 01:44:21.0
6044,ADNI4,052_S_6412,6412,4_m12,m78,2025-05-26,1.0,1.0,07/1951,1951.0,...,1.0,0.0,97024,52,2025-06-04,2025-06-04,v1,e,0.0,2025-06-05 01:44:21.0
6045,ADNI4,168_S_6321,6321,4_init,m72,2024-05-20,1.0,1.0,07/1945,1945.0,...,1.0,1.0,97060,389,2025-06-04,2025-06-04,v1,e,0.0,2025-06-05 01:44:21.0


In [9]:
dataset.columns

Index(['PHASE', 'PTID', 'RID', 'VISCODE', 'VISCODE2', 'VISDATE', 'PTSOURCE',
       'PTGENDER', 'PTDOB', 'PTDOBYY', 'PTHAND', 'PTMARRY', 'PTEDUCAT',
       'PTWORKHS', 'PTWORK', 'PTNOTRT', 'PTRTYR', 'PTHOME', 'PTTLANG',
       'PTPLANG', 'PTADBEG', 'PTCOGBEG', 'PTADDX', 'PTETHCAT', 'PTRACCAT',
       'PTIDENT', 'PTORIENT', 'PTORIENTOT', 'PTENGSPK', 'PTNLANG',
       'PTENGSPKAGE', 'PTCLANG', 'PTLANGSP', 'PTLANGWR', 'PTSPTIM',
       'PTSPOTTIM', 'PTLANGPR1', 'PTLANGSP1', 'PTLANGRD1', 'PTLANGWR1',
       'PTLANGUN1', 'PTLANGPR2', 'PTLANGSP2', 'PTLANGRD2', 'PTLANGWR2',
       'PTLANGUN2', 'PTLANGPR3', 'PTLANGSP3', 'PTLANGRD3', 'PTLANGWR3',
       'PTLANGUN3', 'PTLANGPR4', 'PTLANGSP4', 'PTLANGRD4', 'PTLANGWR4',
       'PTLANGUN4', 'PTLANGPR5', 'PTLANGSP5', 'PTLANGRD5', 'PTLANGWR5',
       'PTLANGUN5', 'PTLANGPR6', 'PTLANGSP6', 'PTLANGRD6', 'PTLANGWR6',
       'PTLANGUN6', 'PTLANGTTL', 'PTETHCATH', 'PTASIAN', 'PTOPI', 'PTBORN',
       'PTBIRPL', 'PTIMMAGE', 'PTIMMWHY', 'PTBIRPR', 'PTBIRGR'

In [ ]:
columns_must_be_verified = ['PTGENDER', 'PTDOB', 'PTEDUCAT']

In [10]:
no_unknow_df = replace_unknown_values(dataset)
no_none_df = handle_null_viscode2(no_unknow_df, columns_must_be_verified, 'VISDATE')
processed_df = drop_if_all_none(no_none_df, columns_must_be_verified)

In [ ]:
processed_df['VISCODE2'] = processed_df['VISCODE2'].apply(lambda x: convert_visitcode_to_int(x))

In [13]:
dataset['VISCODE2'].unique()

array(['sc', 'f', 'm06', 'bl', 'm12', 'm24', 'm18', 'm30', 'm36', 'm42',
       'm48', 'scmri', 'm60', 'm54', 'm03', 'm66', 'm72', 'm84', 'm78',
       'm90', nan, 'm138', 'm210', 'm150', 'm144', 'm216', 'm126', 'm204',
       'm132', 'm156', 'm162', 'm222', 'm228', 'm168', 'm96', 'm174'],
      dtype=object)

np.int64(15)

In [19]:
filtered_df = processed_df[(processed_df['VISCODE2'] == 'sc') | (processed_df['VISCODE2'] == 'f')]
final_df = handle_f_sc_values(filtered_df, processed_df, 'VISCODE2')